In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 
import sys
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # linear algebra 
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
import pydicom 

In [2]:
# Read the data
test = pd.read_csv('../input/siim-isic-melanoma-classification/test.csv')
train = pd.read_csv('../input/siim-isic-melanoma-classification/train.csv')

In [3]:
train.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0


In [4]:
test.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge
0,ISIC_0052060,IP_3579794,male,70.0,NaN
1,ISIC_0052349,IP_7782715,male,40.0,lower extremity
2,ISIC_0058510,IP_7960270,female,55.0,torso
3,ISIC_0073313,IP_6375035,female,50.0,torso
4,ISIC_0073502,IP_0589375,female,45.0,lower extremity


In [5]:
train['sex'].fillna('unknown', inplace=True)
test['sex'].fillna('unknown', inplace=True)

train['age_approx'].fillna(train['age_approx'].mode().values[0], inplace=True)
test['age_approx'].fillna(test['age_approx'].mode().values[0], inplace=True)

train['anatom_site_general_challenge'].fillna('unknown', inplace=True)
test['anatom_site_general_challenge'].fillna('unknown', inplace=True)

In [6]:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()

train['sex_enc'] = enc.fit_transform(train.sex.astype('str'))
test['sex_enc'] = enc.transform(test.sex.astype('str'))

train['age_enc'] = enc.fit_transform(train.age_approx.astype('str'))
test['age_enc'] = enc.transform(test.age_approx.astype('str'))

train['anatom_enc'] = enc.fit_transform(train.anatom_site_general_challenge.astype('str'))
test['anatom_enc'] = enc.transform(test.anatom_site_general_challenge.astype('str'))

In [7]:
train.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,sex_enc,age_enc,anatom_enc
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0,1,8,0
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0,0,8,6
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0,0,9,1
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0,0,8,0
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0,0,10,6


In [8]:
train['age_enc'] = train['age_enc'] / np.mean(train['age_enc'])
test['age_enc'] = test['age_enc'] / np.mean(test['age_enc'])

train['anatom_enc'] = train['anatom_enc'] / np.mean(train['anatom_enc'])
test['anatom_enc'] = test['anatom_enc'] / np.mean(test['anatom_enc'])

train.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,sex_enc,age_enc,anatom_enc
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0,1,0.911943,0.000000
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0,0,0.911943,1.811764
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0,0,1.025936,0.301961
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0,0,0.911943,0.000000
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0,0,1.139929,1.811764


In [9]:
features = [
            'sex_enc',
            'age_enc',
            'anatom_enc'
]

In [10]:
X_train = train[features]
y_train = train['target']

x_test = test[features]

In [11]:
X_train.head()

,sex_enc,age_enc,anatom_enc
0,1,0.911943,0.000000
1,0,0.911943,1.811764
2,0,1.025936,0.301961
3,0,0.911943,0.000000
4,0,1.139929,1.811764


In [12]:
x_test.head()

,sex_enc,age_enc,anatom_enc
0,1,1.459835,1.465908
1,1,0.786065,0.293182
2,0,1.122950,1.172727
3,0,1.010655,1.172727
4,0,0.898360,0.293182


In [13]:
# model tuning

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import time
from sklearn.metrics import roc_auc_score

# A parameter grid for XGBoost
params = {
    'n_estimators':[500],
    'min_child_weight':[4,5], 
    'gamma':[i/10.0 for i in range(3,6)],  
    'subsample':[i/10.0 for i in range(6,11)],
    'colsample_bytree':[i/10.0 for i in range(6,11)], 
    'max_depth': [2,3,4,6,7],
    'objective': ['binary:logistic'],
    'booster': ['gbtree', 'gblinear'],
    'eval_metric': ['rmse'],
    'eta': [i/10.0 for i in range(3,6)],
}

reg = XGBRegressor(n_jobs=-1, nthread=-1)

# run randomized search
n_iter_search = 100
random_search = RandomizedSearchCV(reg, param_distributions=params,
                                   n_iter=n_iter_search, cv=5, iid=False, scoring='roc_auc')

start = time.time()
random_search.fit(X_train, y_train)
print("RandomizedSearchCV took %.2f seconds for %d candidates"
      " parameter settings." % ((time.time() - start), n_iter_search))

[16:21:38] WARNING: /workspace/src/learner.cc:480: 
Parameters: { colsample_bytree, gamma, max_depth, min_child_weight, subsample } might not be used.

  This may not be accurate due to some parameters are only used in language bindings but
  passed down to XGBoost core.  Or some parameters are not used but slip through this
  verification. Please open an issue if you find above cases.


[16:21:41] WARNING: /workspace/src/learner.cc:480: 
Parameters: { colsample_bytree, gamma, max_depth, min_child_weight, subsample } might not be used.

  This may not be accurate due to some parameters are only used in language bindings but
  passed down to XGBoost core.  Or some parameters are not used but slip through this
  verification. Please open an issue if you find above cases.


[16:21:43] WARNING: /workspace/src/learner.cc:480: 
Parameters: { colsample_bytree, gamma, max_depth, min_child_weight, subsample } might not be used.

  This may not be accurate due to some parameters are only used in

/opt/conda/lib/python3.7/site-packages/sklearn/model_selection/_search.py:849: FutureWarning: The parameter 'iid' is deprecated in 0.22 and will be removed in 0.24.
  "removed in 0.24.", FutureWarning


RandomizedSearchCV took 1962.13 seconds for 100 candidates parameter settings.


In [14]:
best_regressor = random_search.best_estimator_
preds = best_regressor.predict(x_test)

In [15]:
sample_submission = pd.read_csv('../input/siim-isic-melanoma-classification/sample_submission.csv')
sample_submission.head()
sample_submission['target'] = preds
sample_submission.to_csv('submission.csv', index=False)